# Lab13 - 影像分割


- 使用 `torchvision.datasets.OxfordIIITPet` 自動下載 Oxford-IIIT Pet dataset。
- 資料預處理寫在 `data_loader.py`：影像縮放為 64x64 後轉成 tensor 並以 ImageNet mean/std 標準化；Ground Truth trimap 使用 nearest resize 保留類別值，並將標籤由 `[1, 2, 3]` 轉為 `[0, 1, 2]`。
- 訓練時對影像與 trimap 套用相同幾何增強，避免標註與圖片錯位。
- 比較 `SegNet`、`SegUNet` 與 `SegMobileUNet` 三種模型。
- 透過 TensorBoard 記錄 loss、mIoU、Precision、Recall 與分割視覺化結果。


## 1. 匯入套件與工具


In [ ]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from lion_pytorch import Lion
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary
from tqdm import tqdm

from data_loader import OxfordPetSegmentationDataset, PetSegmentationTransform, map_trimap
from network import SegNet, SegUNet, SegMobileUNet


## 2. 設定資料集轉換


In [ ]:
dataset_path = './data'

# 幾何轉換由 data_loader 統一處理，確保影像與 trimap 使用相同參數
valid_transform = PetSegmentationTransform((64, 64), train=False)
train_transform = PetSegmentationTransform((64, 64), train=True)
train_transform_color = transforms.ColorJitter(contrast=0.2)


## 3. 評估指標


In [ ]:
def get_confusion_matrix_elements(predicted, target, class_index):
    predicted = predicted.view(-1)
    target = target.view(-1)

    # 針對單一類別轉成二元 mask，方便統計 TP、FP、FN、TN
    predicted_binary = (predicted == class_index).float()
    target_binary = (target == class_index).float()

    TP = torch.sum(predicted_binary * target_binary).item()
    FP = torch.sum(predicted_binary * (1 - target_binary)).item()
    FN = torch.sum((1 - predicted_binary) * target_binary).item()
    TN = torch.sum((1 - predicted_binary) * (1 - target_binary)).item()
    return TP, FP, FN, TN


def performance_metrics(predicted, target, nc=3):
    """計算每個類別的 IoU、Precision、Recall，並回傳平均指標。"""
    ious = []
    precisions = []
    recalls = []

    for class_index in range(nc):
        TP, FP, FN, _ = get_confusion_matrix_elements(predicted, target, class_index)

        iou = TP / (TP + FP + FN) if (TP + FP + FN) != 0 else 0
        precision = TP / (TP + FP) if (TP + FP) != 0 else 0
        recall = TP / (TP + FN) if (TP + FN) != 0 else 0

        ious.append(iou)
        precisions.append(precision)
        recalls.append(recall)

    mIoU = sum(ious) / len(ious)
    mPrecision = sum(precisions) / len(precisions)
    mRecall = sum(recalls) / len(recalls)
    return mIoU, ious, mPrecision, mRecall


## 4. 定義訓練與驗證函式


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, write_img=True, denorm_func=None, device=None):
    model.train()
    running_loss = 0.0
    tqdm_loader = tqdm(dataloader, desc='Training')

    for data in tqdm_loader:
        batch_img, batch_trimap = data
        batch_img = batch_img.to(device)
        batch_trimap = batch_trimap.to(device)

        optimizer.zero_grad()
        outputs = model(batch_img)

        # CrossEntropyLoss 可接受 segmentation logits 與像素類別 mask
        loss = criterion(outputs.flatten(2), batch_trimap.flatten(1)).mean()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        tqdm_loader.set_postfix({'Loss': loss.item()})

    if write_img:
        # 將影像、真實 trimap、預測 trimap 並排，方便在 TensorBoard 檢查分割品質
        outputs = torch.argmax(outputs, 1)
        outputs = map_trimap(outputs, False) / 255.
        batch_trimap = map_trimap(batch_trimap, False) / 255.
        outputs = outputs.unsqueeze(1).repeat(1, 3, 1, 1)
        batch_trimap = batch_trimap.unsqueeze(1).repeat(1, 3, 1, 1)
        batch_img = denorm_func(batch_img)
        debug_image = torch.cat([batch_img, batch_trimap, outputs], 2)
        debug_image = torchvision.utils.make_grid(debug_image[:25])
    else:
        debug_image = None

    return running_loss / len(dataloader), debug_image


def validate(model, dataloader, criterion, device=None):
    model.eval()
    running_loss = 0.0
    tqdm_loader = tqdm(dataloader, desc='Validation')
    preds, targets = [], []

    for data in tqdm_loader:
        batch_img, batch_trimap = data
        batch_img = batch_img.to(device)
        batch_trimap = batch_trimap.to(device)

        with torch.no_grad():
            outputs = model(batch_img)
            loss = criterion(outputs, batch_trimap).mean()
            running_loss += loss.item()
            preds.append(torch.argmax(outputs, 1).cpu())
            targets.append(batch_trimap.cpu())

        tqdm_loader.set_postfix({'Loss': loss.item()})

    miou, ious, mprecision, mrecall = performance_metrics(torch.cat(preds), torch.cat(targets))
    return running_loss / len(dataloader), miou, ious, mprecision, mrecall

## 5. 設定訓練參數


In [ ]:
def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'


seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

batch_size = 128
learning_rate = 0.0005
num_epochs = 50


## 6. 建立資料集與 DataLoader


In [ ]:
# torchvision 會自動下載並建立 trainval/test split，已存在時不會重複下載
train_dataset = OxfordPetSegmentationDataset(dataset_path, 'trainval', train_transform, train_transform_color)
valid_dataset = OxfordPetSegmentationDataset(dataset_path, 'test', valid_transform)

n_workers = 2 if os.name == 'nt' else 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=n_workers)
valid_loader = DataLoader(valid_dataset, batch_size=int(batch_size * 1.5), shuffle=True, num_workers=n_workers)
denorm_func = train_dataset.denorm



## 7. 訓練模型


In [ ]:
model_segnet = SegNet(3)
model_segUnet = SegUNet(3)
model_segMobileUnet = SegMobileUNet(3)

model_seq = list(zip(
    ['SegNet', 'SegUNet', 'SegMobileUNet'],
    [model_segnet, model_segUnet, model_segMobileUnet]
))

for model_name, model in model_seq:
    print(f'Training {model_name}')
    model.to(device)

    train_loss = []
    val_loss = []
    criterion = nn.CrossEntropyLoss(reduction='none')
    optimizer = Lion(model.parameters(), lr=learning_rate)
    writer = SummaryWriter(f'runs/{model_name}')

    dummy_input = torch.rand((1, 3, 64, 64)).to(device)
    writer.add_graph(model, dummy_input)
    summary(model, input_data=dummy_input)

    for epoch in range(num_epochs):
        epoch_train_loss, debug_img = train_epoch(model, train_loader, criterion, optimizer, True, denorm_func, device)
        epoch_val_loss, *valid_metric = validate(model, valid_loader, criterion, device)

        train_loss.append(epoch_train_loss)
        val_loss.append(epoch_val_loss)

        print(f'Epoch {epoch + 1} - Training Loss: {epoch_train_loss:.4f}, Validation Loss: {epoch_val_loss:.4f}')

        if epoch % 5 == 0:
            # 觀察權重分布變化，可協助判斷訓練是否不穩定或梯度異常
            for name, param in model.named_parameters():
                if param.requires_grad and 'bias' not in name:
                    writer.add_histogram(name, param, epoch)

        writer.add_image('debug_image', debug_img, epoch)
        writer.add_scalar('train/loss', epoch_train_loss, epoch)
        writer.add_scalar('valid/loss', epoch_val_loss, epoch)

        miou, ious, mprecision, mrecall = valid_metric
        writer.add_scalar('valid/mIoU', miou, epoch)
        writer.add_scalar('valid/mPrecision', mprecision, epoch)
        writer.add_scalar('valid/mRecall', mrecall, epoch)
        for i, iou in enumerate(ious):
            writer.add_scalar(f'valid/IoU-{i}', iou, epoch)

    writer.close()



## 8. TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs
